# LOGAN: why Prodigal did not call the candidates

**Draft caption.** Prodigal 2.6.3 (-p meta) rerun on the query contigs with -s, which lists every potential gene and its scores; the rerun reproduces all original calls. For each query, the best-scoring potential gene ending at its stop. (a) Coding score; (b) total score (coding + start + RBS + upstream); (c) ORF GC minus the GC of the Prodigal model chosen for the contig; values outside each axis range are clipped to its edges. (d) Why the ORF was not called: no potential gene at that stop, negative coding score, positive coding score but start penalties make the total negative, or positive total but not chosen. Bacterial-hit candidates: best full-length target bacterial.

**Outputs:** `../images/figL09_prodigal_skip.pdf` (vector) and `../images/figL09_prodigal_skip.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL09_prodigal_skip.tsv.gz` (52,834 rows; sha256 `e096bf02c944…`): Prodigal -s best potential gene at each query stop; one row per query. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates/skip/skip_features.parquet`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL09_prodigal_skip.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
F = pd.read_csv(DATA / "figL09_prodigal_skip.tsv.gz", sep="\t")
GROUPS = ["Prodigal gene, bacterial hit", "candidate, bacterial hit", "candidate, other hit",
          "candidate, no hit", "low-score intergenic"]
GCOL = dict(zip(GROUPS, [SLOTS[0], SLOTS[1], SLOTS[4], SLOTS[3], SLOTS[2]]))
REASONS = ["positive, not called", "start penalties", "negative coding", "no potential gene"]
RCOL = dict(zip(REASONS, [SLOTS[0], SLOTS[3], SLOTS[1], GRID]))
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (col, lab, lim), letter in zip(axes.flat[:3], (("CodPot", "Prodigal coding score (best start)", (-30, 60)),
                                                       ("Total", "Prodigal total score (best start)", (-40, 60)),
                                                       ("orf_minus_model_gc", "ORF GC − Prodigal model GC", (-.3, .3))),
                                       "abc"):
    for g in GROUPS:
        v = F.loc[F.group == g, col].dropna().clip(*lim)
        if len(v) > 20:
            sns.kdeplot(v, ax=ax, color=GCOL[g], label=f"{g} (n={len(v):,})", lw=2, cut=0)
    ax.axvline(0, color=INK2, lw=1, ls="--")
    ax.set_xlabel(lab)
    ax.set_xlim(*lim)
    panel(ax, letter)
axes[0, 0].legend(fontsize=8.5, loc="upper right")
ax = axes[1, 1]
cg = GROUPS[1:]
R = pd.crosstab(F.group, F.reason, normalize="index").reindex(index=cg, columns=REASONS, fill_value=0)
left = np.zeros(len(cg))
for rs in REASONS:
    ax.barh(range(len(cg)), R[rs], left=left, color=RCOL[rs], label=rs, height=0.6, edgecolor="white", lw=2)
    left += R[rs].to_numpy()
ax.set_yticks(range(len(cg)), cg)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("why Prodigal did not call the ORF")
ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.4, 1.0), fontsize=9)
panel(ax, "d", x=-0.4)
fig.tight_layout()
save(fig, "figL09_prodigal_skip")